# 1 - Instalação payspark, início de seção e conexão

In [1]:
!pip install pyspark
from google.colab import drive
from pyspark.sql import SparkSession

In [2]:
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
memory_limit = "12g"
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Hackathon_Silver-Gold") \
    .config("spark.driver.memory", memory_limit) \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.executor.memory", memory_limit) \
    .config("spark.memory.fraction", "0.8") \
    .getOrCreate()

#

# 2 - Bibliotecas

In [4]:
import os
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import *

#

# 3 - Funções

In [5]:
# --------------------------
# Função para retornar o shape
# --------------------------

def get_shape(dataframe):
    linhas = f'Quanitade de linhas: {dataframe.count()}'
    colunas = f'Quanitade de Colunas: {len(dataframe.columns)}'
    return linhas, colunas

In [8]:
# --------------------------
# Função para retornar tabela agrupada e percentual
# --------------------------
def Freq(pTabela,pColuna, pTop = 1000):

    qtd_total=pTabela.count()

    pTabela.registerTempTable("tab_input")
    frq = spark.sql(
            """
                select
                    {col},
                    count(*) as qtd_absoluto,
                    round(100*(count(*) / {tot}),2) as qtd_percentual
                from
                    tab_input
                group by
                    {col}
                order by
                    2 desc
            """.format(col=pColuna, tot=qtd_total))

    qtd=frq.count()
    print('Quantidade de dominios',qtd)
    if qtd > 500:
        frq.show(pTop,truncate=False)
        return "Dominio muito granular"

    else:
        frq.show(qtd,truncate=False)
        print("volumetria total:",qtd_total)
        return 'Freq da coluna ' + pColuna;

In [9]:
def comparar_datasets(df1, df2, variavel_alvo):
    """
    Compara o shape e a proporção de uma variável (ex: FPD) entre dois DataFrames.
    """
    # 1. Estatísticas do Primeiro DataFrame
    shape1_rows = df1.count()
    shape1_cols = len(df1.columns)
    # Calcula a média (proporção) da variável alvo
    prop1 = df1.select(F.avg(F.col(variavel_alvo).cast("float"))).collect()[0][0]
    prop1_perc = (prop1 or 0) * 100

    # 2. Estatísticas do Segundo DataFrame
    shape2_rows = df2.count()
    shape2_cols = len(df2.columns)
    prop2 = df2.select(F.avg(F.col(variavel_alvo).cast("float"))).collect()[0][0]
    prop2_perc = (prop2 or 0) * 100

    # 3. Cálculos de Diferença
    diff_rows = shape1_rows - shape2_rows
    perc_perda_rows = (diff_rows / shape1_rows) * 100 if shape1_rows != 0 else 0
    diff_prop = prop1_perc - prop2_perc

    # 4. Exibição dos Resultados
    print(f"{'='*50}")
    print(f" ANÁLISE COMPARATIVA: {variavel_alvo.upper()}")
    print(f"{'='*50}")

    print(f" DATAFRAME 1 (Referência):")
    print(f"   - Shape: ({shape1_rows}, {shape1_cols})")
    print(f"   - Proporção de {variavel_alvo}: {prop1_perc:.2f}%")

    print(f"\n DATAFRAME 2 (Atual):")
    print(f"   - Shape: ({shape2_rows}, {shape2_cols})")
    print(f"   - Proporção de {variavel_alvo}: {prop2_perc:.2f}%")

    print(f"\n{'='*50}")
    print(f" IMPACTO DA TRANSFORMAÇÃO:")
    print(f"   - Linhas removidas: {diff_rows} ({perc_perda_rows:.2f}% de perda)")
    print(f"   - Variação no Target: {diff_prop:.4f} p.p. (pontos percentuais)")
    print(f"{'='*50}")

#

# 4 - Carregando os Dados

In [10]:
# Pastas no drive
pastas_alvo = [
    'tabela_bi_bi_dim_status_plataforma',
    'tabela_bi_dim_canal_aquisicao_credito',
    'tabela_bi_dim_forma_pagamento',
    'tabela_bi_dim_instituicao',
    'tabela_bi_dim_plano_preco',
    'tabela_bi_dim_plataforma',
    'tabela_bi_dim_promocao_credito',
    'tabela_bi_dim_tecnologia',
    'tabela_bi_dim_tipo_credito',
    'tabela_bi_dim_tipo_insercao',
    'tabela_bi_dim_tipo_recarga',
    'tabela_cadastral',
    'tabela_pagamento',
    'tabela_recarga',
    'tabela_score_bureau_full',
    'tabela_telco'
]

In [11]:
# Endereço no Drive e Endereço da pasta local
base_drive = "/content/gdrive/MyDrive/Hackathon_POD/Silver/Silver"
base_local = "/content/dados_locais"

In [12]:
# --------------------------
# Carregando os dados para a memoria local do colab
# --------------------------

for pasta in pastas_alvo:
  origem = os.path.join(base_drive, pasta)
  destino = os.path.join(base_local, pasta)

  if os.path.exists(origem):
    print(f'Procecando pasta {pasta}..')

    os.makedirs(destino, exist_ok = True)

    !cp -rn '{origem}/.' '{destino}'
    print(f'Sucesso, Copiado para: {destino}')
  else:
    print(f'ERRO: Pasta não encontrada no Drive: {origem}')

print('\n--PROCESSO FINALIZADO--')
print(f'Pastas criadas localmente: {os.listdir(base_local)}')

Procecando pasta tabela_bi_bi_dim_status_plataforma..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_bi_dim_status_plataforma
Procecando pasta tabela_bi_dim_canal_aquisicao_credito..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_canal_aquisicao_credito
Procecando pasta tabela_bi_dim_forma_pagamento..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_forma_pagamento
Procecando pasta tabela_bi_dim_instituicao..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_instituicao
Procecando pasta tabela_bi_dim_plano_preco..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_plano_preco
Procecando pasta tabela_bi_dim_plataforma..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_plataforma
Procecando pasta tabela_bi_dim_promocao_credito..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_promocao_credito
Procecando pasta tabela_bi_dim_tecnologia..
Sucesso, Copiado para: /content/dados_locais/tabela_bi_dim_tecnologia
Procecando pasta

In [13]:
# --------------------------
# Carregando os dados para um unico dicionario
# --------------------------
dfs = {}
path_base = "/content/dados_locais"

print('--- Carregando Bases Parquet ---')

for i in pastas_alvo:

  print(f'Carregado bases Parquet:{i}')
  dfs[i] = spark.read.parquet(f'{path_base}/{i}/*.parquet')

print(dfs.keys())

--- Carregando Bases Parquet ---
Carregado bases Parquet:tabela_bi_bi_dim_status_plataforma
Carregado bases Parquet:tabela_bi_dim_canal_aquisicao_credito
Carregado bases Parquet:tabela_bi_dim_forma_pagamento
Carregado bases Parquet:tabela_bi_dim_instituicao
Carregado bases Parquet:tabela_bi_dim_plano_preco
Carregado bases Parquet:tabela_bi_dim_plataforma
Carregado bases Parquet:tabela_bi_dim_promocao_credito
Carregado bases Parquet:tabela_bi_dim_tecnologia
Carregado bases Parquet:tabela_bi_dim_tipo_credito
Carregado bases Parquet:tabela_bi_dim_tipo_insercao
Carregado bases Parquet:tabela_bi_dim_tipo_recarga
Carregado bases Parquet:tabela_cadastral
Carregado bases Parquet:tabela_pagamento
Carregado bases Parquet:tabela_recarga
Carregado bases Parquet:tabela_score_bureau_full
Carregado bases Parquet:tabela_telco
dict_keys(['tabela_bi_bi_dim_status_plataforma', 'tabela_bi_dim_canal_aquisicao_credito', 'tabela_bi_dim_forma_pagamento', 'tabela_bi_dim_instituicao', 'tabela_bi_dim_plano_preco

#

# 5 - Score_1 e Score_2

In [ ]:
df = dfs['tabela_score_bureau_full']

In [ ]:
df.show(5)

+-----------------+--------------+-----------+------+---------------+----+----+------------+--------+----------------+--------+----------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|FLAG_INSTALACAO|PROD| FPD|   flag_mig2|SCORE_01|SCORE_01_MISSING|SCORE_02|SCORE_02_MISSING|
+-----------------+--------------+-----------+------+---------------+----+----+------------+--------+----------------+--------+----------------+
|ZZZZZZZX7T9202410|          true|ZZZZZZZX7T9|202410|              1| CMV|   1|   Aquisição|       2|               0|       1|               0|
|ZZZZZZZ8TZ8202410|          true|ZZZZZZZ8TZ8|202410|              0| CMV|NULL|SEM_MIGRACAO|     562|               0|     559|               0|
|ZZZZZZW9XWN202410|         false|ZZZZZZW9XWN|202410|              0| CMV|NULL|SEM_MIGRACAO|     585|               0|     559|               0|
|ZZZZZX7XWY8202410|         false|ZZZZZX7XWY8|202410|              1| CMV|   0|         PRE|     562|               0|     636|   

In [ ]:
#-------------------------------------------------
# REMOVENDO CLIENTES QUE NÃO CONTRATARAM (SEM FPD)
#-------------------------------------------------
df = df.dropna(subset = ['FPD'])

In [ ]:
df.where(F.col('FPD').isNull()).show()

+--------+--------------+-------+-----+---------------+----+---+---------+--------+----------------+--------+----------------+
|ID_UNICO|GRUPO_CONTROLE|NUM_CPF|SAFRA|FLAG_INSTALACAO|PROD|FPD|flag_mig2|SCORE_01|SCORE_01_MISSING|SCORE_02|SCORE_02_MISSING|
+--------+--------------+-------+-----+---------------+----+---+---------+--------+----------------+--------+----------------+
+--------+--------------+-------+-----+---------------+----+---+---------+--------+----------------+--------+----------------+



In [ ]:
# ----------------------------
# IDENTIFICANDO OOT COM FLAG
#-----------------------------
df = df.withColumn(
    'Flag_OOT',
    F.when(F.col('SAFRA').isin('202502', '202503'), 1).otherwise(0)
)

In [ ]:
Freq(df,'PROD')

/usr/local/lib/python3.12/dist-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


Quantidade de dominios 1
+----+------------+--------------+
|PROD|qtd_absoluto|qtd_percentual|
+----+------------+--------------+
|CMV |2633900     |100.0         |
+----+------------+--------------+

volumetria total: 2633900


'Freq da coluna PROD'

In [ ]:
# ------------------------------------------------------
# REMOÇÃO DA VARIAVEL PROD POR POSSUIR SOMENTE UM VALOR
#-------------------------------------------------------
df = df.drop('PROD')

In [ ]:
Freq(df, 'flag_mig2')

Quantidade de dominios 3
+---------+------------+--------------+
|flag_mig2|qtd_absoluto|qtd_percentual|
+---------+------------+--------------+
|Aquisição|1338888     |50.83         |
|PRE      |1290526     |49.0          |
|FLEX     |4486        |0.17          |
+---------+------------+--------------+

volumetria total: 2633900


'Freq da coluna flag_mig2'

In [ ]:
#------------------------------------------
# APLICAÇÃO DE OHE NA VARIÁVEL 'flag_mig2'
#------------------------------------------
df = df.withColumns({
    'MIG_Aquisicao': F.when(F.col('flag_mig2') == 'Aquisição', 1).otherwise(0),
    'MIG_PRE': F.when(F.col('flag_mig2') == 'PRE', 1).otherwise(0),
    'MIG_FLEX': F.when(F.col('flag_mig2') == 'FLEX', 1).otherwise(0)
}).drop('flag_mig2')

In [ ]:
#-------------------------------------------------------------------
# NORMALIZANDO A VARIAVEL 'GRUPO_CONTROLE' PARA O PADRÃO DO DF (int)
#-------------------------------------------------------------------
df = df.withColumn(
    'GRUPO_CONTROLE',
    F.col('GRUPO_CONTROLE').cast('int')
)

In [ ]:
ordem = ['ID_UNICO','GRUPO_CONTROLE','NUM_CPF', 'SAFRA', 'Flag_OOT', 'MIG_Aquisicao' , 'MIG_PRE', 'MIG_FLEX',
         'FPD', 'SCORE_01', 'SCORE_01_MISSING', 'SCORE_02', 'SCORE_02_MISSING']

In [ ]:
#-----------------------
# REORDENANDO AS COLUNAS
#-----------------------
df = df[ordem]

In [ ]:
df.show(5)

+-----------------+--------------+-----------+------+--------+-------------+-------+--------+---+--------+----------------+--------+----------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|Flag_OOT|MIG_Aquisicao|MIG_PRE|MIG_FLEX|FPD|SCORE_01|SCORE_01_MISSING|SCORE_02|SCORE_02_MISSING|
+-----------------+--------------+-----------+------+--------+-------------+-------+--------+---+--------+----------------+--------+----------------+
|ZZZZZZZX7T9202410|             1|ZZZZZZZX7T9|202410|       0|            1|      0|       0|  1|       2|               0|       1|               0|
|ZZZZZX7XWY8202410|             0|ZZZZZX7XWY8|202410|       0|            0|      1|       0|  0|     562|               0|     636|               0|
|ZZZZZX8TTUZ202410|             0|ZZZZZX8TTUZ|202410|       0|            1|      0|       0|  1|     538|               0|     570|               0|
|ZZZZZX88YXY202410|             0|ZZZZZX88YXY|202410|       0|            0|      1|       0|  1|   

In [ ]:
df.select('SCORE_01', 'SCORE_02').describe().show()

+-------+-----------------+------------------+
|summary|         SCORE_01|          SCORE_02|
+-------+-----------------+------------------+
|  count|          2633900|           2633900|
|   mean|586.9768297961198| 650.5487892478834|
| stddev|74.06244093379864|103.15780310256353|
|    min|               -1|                -1|
|    max|              778|               926|
+-------+-----------------+------------------+



Durante a ciração dos books foi percebeu-se que a Claro adota um padrão de sentinélas:
* -1: Não se Aplica
* -2: Não determinado
* -3: Não informado

Sendo assim foi adotado uma nova categoria para imputar valores nulos em que a natureza deles é desconhecida:
* -4: Desconhecido

In [ ]:
#--------------------------------------------
# ALTERANDO PADRÃO DOS VALORES SENTINELAS
#-------------------------------------------
mapeamento = {-1: -4}
df = df.replace(mapeamento, subset=['SCORE_01', 'SCORE_02'])

In [ ]:
df.select('SCORE_01', 'SCORE_02').describe().show()

+-------+-----------------+-----------------+
|summary|         SCORE_01|         SCORE_02|
+-------+-----------------+-----------------+
|  count|          2633900|          2633900|
|   mean|586.9567014693041|650.5466524925016|
| stddev| 74.2224704890228|103.1713291010178|
|    min|               -4|               -4|
|    max|              778|              926|
+-------+-----------------+-----------------+



In [ ]:
df.printSchema()

root
 |-- ID_UNICO: string (nullable = true)
 |-- GRUPO_CONTROLE: integer (nullable = true)
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: string (nullable = true)
 |-- Flag_OOT: integer (nullable = false)
 |-- MIG_Aquisicao: integer (nullable = false)
 |-- MIG_PRE: integer (nullable = false)
 |-- MIG_FLEX: integer (nullable = false)
 |-- FPD: string (nullable = true)
 |-- SCORE_01: integer (nullable = true)
 |-- SCORE_01_MISSING: integer (nullable = true)
 |-- SCORE_02: integer (nullable = true)
 |-- SCORE_02_MISSING: integer (nullable = true)



In [ ]:
#------------------------------------------------------------------------------------------------------
# SEPARANDO OS DATASETS EM score_01 SOMENTE COM AS INFORMAÇÕES DO SCORE_01 score_01_02 COM AMBOS SCORES
#-------------------------------------------------------------------------------------------------------
score_01 = df.drop('SCORE_02', 'SCORE_02_MISSING')
score_01_02 = df

In [ ]:
score_01.show(5)
score_01_02.show(5)

+-----------------+--------------+-----------+------+--------+-------------+-------+--------+---+--------+----------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|Flag_OOT|MIG_Aquisicao|MIG_PRE|MIG_FLEX|FPD|SCORE_01|SCORE_01_MISSING|
+-----------------+--------------+-----------+------+--------+-------------+-------+--------+---+--------+----------------+
|ZZZZZZZX7T9202410|             1|ZZZZZZZX7T9|202410|       0|            1|      0|       0|  1|       2|               0|
|ZZZZZX7XWY8202410|             0|ZZZZZX7XWY8|202410|       0|            0|      1|       0|  0|     562|               0|
|ZZZZZX8TTUZ202410|             0|ZZZZZX8TTUZ|202410|       0|            1|      0|       0|  1|     538|               0|
|ZZZZZX88YXY202410|             0|ZZZZZX88YXY|202410|       0|            0|      1|       0|  1|     546|               0|
|ZZZZZYT7XYT202410|             0|ZZZZZYT7XYT|202410|       0|            0|      1|       0|  0|     621|               0|
+-------

In [ ]:
#----------------------------------------------------------
# COMPARAÇÃO ENTRE O DF ORIGINA E O QUE FOI PARA A BASELINE
#----------------------------------------------------------
comparar_datasets(df, score_01_02, 'FPD')

 ANÁLISE COMPARATIVA: FPD
 DATAFRAME 1 (Referência):
   - Shape: (2633900, 13)
   - Proporção de FPD: 21.23%

 DATAFRAME 2 (Atual):
   - Shape: (2633900, 13)
   - Proporção de FPD: 21.23%

 IMPACTO DA TRANSFORMAÇÃO:
   - Linhas removidas: 0 (0.00% de perda)
   - Variação no Target: 0.0000 p.p. (pontos percentuais)


## Salvamento

In [ ]:
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Gold/score_01"
score_01.write.mode("overwrite").parquet(path_silver)

In [ ]:
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Gold/score_02"
score_01_02.write.mode("overwrite").parquet(path_silver)

## Relatório

Para o bodelo baseline foi retirado da base score_bureau_full os cliente que nao contrataram (FPD = null). e para a visão increlemntal do KS ela foi dividade em dois dataframes:

**- 1º score_01:**
* `ID_UNICO`: Chave primária composta (Concatenação de CPF + Safra). Garante a unicidade do registro para cada período de análise.
* `GRUPO_CONTROLE`: Variável binária que identifica clientes pertencentes ao grupo controle, utilizada para medir o impacto incremental de políticas de crédito.
* `NUM_CPF`: Número de identificação fiscal do cliente, devidamente anonimizado para conformidade com normas de privacidade de dados.
* `SAFRA` : Identificador do mês de referência do evento de crédito
* `Flag_OOT`: Indicador de Out-of-Time Validation. Identifica as safras reservadas para a validação final do modelo, simulando o desempenho em dados futuros que não participaram do treinamento.
* `MIG_Aquisicao`: Indica se i cliente migrou ou não no formato aquisição
* `MIG_PRE`: Indica se o cliente migrou ou não no formato PRÉ.
* `MIG_FLEX`: Indica se o cliente migrou ou não no formato.
* `FPD` (First Payment Default):Variável dependente (target) do modelo. Indica se o cliente deixou de realizar o primeiro pagamento da fatura após a migração. {1: Inadimplente, 0: Adimplente}.
* `SCORE_01`: Valor numérico do primeiro Score de Crédito fornecido pelo Bureau. Utilizado como um preditor externo de probabilidade de inadimplência.
* `SCORE_01_MISSING`: Flag binária que identifica a ausência do dado original no Bureau {1: Ausente, 0: Presente}.

**- 2º score_01_02** Adicionado a score_01:
* `SCORE_02`:Valor numérico do segundo Score de Crédito fornecido pelo Bureau.
* `SCORE_02_MISSING`: Flag binária que identifica a ausência do dado original no Bureau {1: Ausente, 0: Presente}.

**`OBS: `** Valores -4 nessas variaveis são sentinélas que identificam quando o valor veio ausente da base original.


#

# 6 - Telco

In [ ]:
df = dfs['tabela_telco']

In [ ]:
df.show(5)

+-----------------+--------------+-----------+------+---------------+---+---------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------+---+---+---+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|FLAG_INSTALACAO|FPD|flag_mig2|var_26|var_27|var_28|var_29|var_30|var_31|var_32|var_33|var_34|var_35|var_36|var_37|var_38|var_39|var_40|var_41|var_42|var_43|var_44|var_45|var_46|var_47|var_48|var_49|var_50|var_51|var_52|var_53|var_54|var_55|var_56|var_57|var_58|var_59|var_60|var_61|var_62|var_63|var_64|var_65|var_66|var_67|var_68|var_69|var_70|var_71|

In [ ]:
Freq(df, 'PROD')

Quantidade de dominios 3
+----+------------+--------------+
|PROD|qtd_absoluto|qtd_percentual|
+----+------------+--------------+
|CMV |1346917     |98.52         |
|NET |16209       |1.19          |
|DTH |3978        |0.29          |
+----+------------+--------------+

volumetria total: 1367104


'Freq da coluna PROD'

In [ ]:
#--------------------------------
# APLICANDO OHT NA COLUNA PROD
#--------------------------------
df = df.withColumns({
    'CMV': F.when(F.col('PROD') == 'CMV', 1).otherwise(0),
    'NET': F.when(F.col('PROD') == 'NET', 1).otherwise(0),
    'DTH': F.when(F.col('PROD') == 'DTH', 1).otherwise(0)
}).drop('PROD')

In [ ]:
Freq(df, 'flag_mig2')

Quantidade de dominios 4
+------------+------------+--------------+
|flag_mig2   |qtd_absoluto|qtd_percentual|
+------------+------------+--------------+
|PRE         |1290526     |94.4          |
|SEM_MIGRACAO|58130       |4.25          |
|Aquisição   |18362       |1.34          |
|FLEX        |86          |0.01          |
+------------+------------+--------------+

volumetria total: 1367104


'Freq da coluna flag_mig2'

In [ ]:
#--------------------------------------
# ADICIONANDO COLUNA 'MIG_SEM_MIGRACAO'
#--------------------------------------
df = df.withColumn(
    'MIG_SEM_MIGRACAO',
    F.when(F.col('flag_mig2') == 'SEM_MIGRACAO', 1).otherwise(0)
)

In [ ]:
numer = ['var_26', 'var_27', 'var_28', 'var_29', 'var_30', 'var_31', 'var_32', 'var_33', 'var_34', 'var_35', 'var_36', 'var_37', 'var_38', 'var_39', 'var_40',
         'var_41', 'var_42', 'var_43', 'var_44', 'var_45', 'var_46', 'var_47', 'var_48', 'var_49', 'var_50', 'var_51', 'var_52', 'var_53', 'var_54', 'var_55',
         'var_56', 'var_57', 'var_58', 'var_59', 'var_60', 'var_61', 'var_62', 'var_63', 'var_64', 'var_65', 'var_66', 'var_67', 'var_68', 'var_69', 'var_70',
         'var_71', 'var_72', 'var_73', 'var_74', 'var_75', 'var_76', 'var_77', 'var_78', 'var_79', 'var_80', 'var_81', 'var_82', 'var_83', 'var_84', 'var_85',
         'var_86', 'var_87', 'var_88', 'var_89', 'var_90', 'var_91', 'var_92', 'var_93']

In [ ]:
df.select('var_46').describe().show()

+-------+------------------+
|summary|            var_46|
+-------+------------------+
|  count|           1367104|
|   mean|         72.961900|
| stddev|120.85795645680064|
|    min|           -999.00|
|    max|            308.00|
+-------+------------------+



In [ ]:
#--------------------------------------------
# ALTERANDO PADRÃO DOS VALORES SENTINELAS
#-------------------------------------------
df = df.na.replace(-999, -4, subset=numer)

In [ ]:
df.select('var_46').describe().show()

+-------+------------------+
|summary|            var_46|
+-------+------------------+
|  count|           1367104|
|   mean|         73.904422|
| stddev|116.28788003047461|
|    min|             -4.00|
|    max|            308.00|
+-------+------------------+



In [ ]:
df.printSchema()

root
 |-- ID_UNICO: string (nullable = true)
 |-- GRUPO_CONTROLE: boolean (nullable = true)
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: string (nullable = true)
 |-- FLAG_INSTALACAO: string (nullable = true)
 |-- FPD: string (nullable = true)
 |-- flag_mig2: string (nullable = true)
 |-- var_26: decimal(18,2) (nullable = true)
 |-- var_27: decimal(18,2) (nullable = true)
 |-- var_28: decimal(18,2) (nullable = true)
 |-- var_29: decimal(18,2) (nullable = true)
 |-- var_30: decimal(18,2) (nullable = true)
 |-- var_31: decimal(18,2) (nullable = true)
 |-- var_32: decimal(18,2) (nullable = true)
 |-- var_33: decimal(18,2) (nullable = true)
 |-- var_34: decimal(18,2) (nullable = true)
 |-- var_35: decimal(18,2) (nullable = true)
 |-- var_36: decimal(18,2) (nullable = true)
 |-- var_37: decimal(18,2) (nullable = true)
 |-- var_38: decimal(18,2) (nullable = true)
 |-- var_39: decimal(18,2) (nullable = true)
 |-- var_40: decimal(18,2) (nullable = true)
 |-- var_41: decimal(18,2) (nullab

In [ ]:
#-------------------------------------------------------------------
# NORMALIZANDO A VARIAVEL 'GRUPO_CONTROLE' PARA O PADRÃO DO DF (int)
#-------------------------------------------------------------------
df = df.withColumns({
    'GRUPO_CONTROLE': F.col('GRUPO_CONTROLE').cast('int'),
    'FPD': F.col('FPD').cast('int')
})

In [ ]:
df.select('GRUPO_CONTROLE', 'FPD').printSchema()

root
 |-- GRUPO_CONTROLE: integer (nullable = true)
 |-- FPD: integer (nullable = true)



In [ ]:
df.show(5)

+-----------------+--------------+-----------+------+---------------+---+---------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------+---+---+---+----------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|FLAG_INSTALACAO|FPD|flag_mig2|var_26|var_27|var_28|var_29|var_30|var_31|var_32|var_33|var_34|var_35|var_36|var_37|var_38|var_39|var_40|var_41|var_42|var_43|var_44|var_45|var_46|var_47|var_48|var_49|var_50|var_51|var_52|var_53|var_54|var_55|var_56|var_57|var_58|var_59|var_60|var_61|var_62|var_63|var_64|var_65|var_66|var_67|var_68|var_

In [ ]:
ordem = ['ID_UNICO', 'NUM_CPF', 'SAFRA','MIG_SEM_MIGRACAO', 'CMV', 'NET', 'DTH', 'FPD',
         'var_26', 'var_27', 'var_28', 'var_29', 'var_30', 'var_31', 'var_32', 'var_33', 'var_34', 'var_35', 'var_36', 'var_37', 'var_38', 'var_39', 'var_40',
         'var_41', 'var_42', 'var_43', 'var_44', 'var_45', 'var_46', 'var_47', 'var_48', 'var_49', 'var_50', 'var_51', 'var_52', 'var_53', 'var_54', 'var_55',
         'var_56', 'var_57', 'var_58', 'var_59', 'var_60', 'var_61', 'var_62', 'var_63', 'var_64', 'var_65', 'var_66', 'var_67', 'var_68', 'var_69', 'var_70',
         'var_71', 'var_72', 'var_73', 'var_74', 'var_75', 'var_76', 'var_77', 'var_78', 'var_79', 'var_80', 'var_81', 'var_82', 'var_83', 'var_84', 'var_85',
         'var_86', 'var_87', 'var_88', 'var_89', 'var_90', 'var_91', 'var_92', 'var_93', 'var26_a_var77_MISSING']

In [ ]:
telco = df[ordem]

In [ ]:
telco.show(5)

+-----------------+-----------+------+----------------+---+---+---+---+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------+
|         ID_UNICO|    NUM_CPF| SAFRA|MIG_SEM_MIGRACAO|CMV|NET|DTH|FPD|var_26|var_27|var_28|var_29|var_30|var_31|var_32|var_33|var_34|var_35|var_36|var_37|var_38|var_39|var_40|var_41|var_42|var_43|var_44|var_45|var_46|var_47|var_48|var_49|var_50|var_51|var_52|var_53|var_54|var_55|var_56|var_57|var_58|var_59|var_60|var_61|var_62|var_63|var_64|var_65|var_66|var_67|var_68|var_69|var_70|var_71|var_72| var_73|var_74|var_75|var_76|

In [ ]:
#--------------------------------------------------------
#REALIZANDO LEFET JOIN DA tabela_telco COM a score_01_02
#--------------------------------------------------------


colunas_base = score_01_02.columns
colunas_telco = telco.columns

colunas_para_trazer = [c for c in colunas_telco if c not in colunas_base or c == 'ID_UNICO']

df_telco_recorte = telco.select(colunas_para_trazer)

# Left Join
# Base Esquerda: Score
# Base Direita: Telco
df_book_v3 = score_01_02.join(df_telco_recorte, on='ID_UNICO', how='left')


vars_novas_geral = [c for c in colunas_para_trazer if c != 'ID_UNICO']

# Regra A: Começa com 'var' -> Imputar -1 'Não se Aplica'
vars_prefixo_var = [c for c in vars_novas_geral if c.startswith('var')]

# Regra B: O restante (Net, DTH, CMV, etc) -> Imputar 0
vars_outras = [c for c in vars_novas_geral if not c.startswith('var')]


# 5. Aplicação dos Fillna
# Aplicar -1 (Não se Aplica) nas variaveis anonimizadas (var_...)
if vars_prefixo_var:
    df_book_v3 = df_book_v3.fillna(-1, subset=vars_prefixo_var)

# Aplicar 0 nas demais numéricas (Produtos, CMV, etc)
if vars_outras:
    df_book_v3 = df_book_v3.fillna(0, subset=vars_outras)

print(f"Total de Linhas: {df_book_v3.count()}") # Valor tem que dar: 2633900
print(f"Total de Colunas: {len(df_book_v3.columns)}")

Total de Linhas: 2633900
Total de Colunas: 86


In [ ]:
#------------------------------------------------------------------------
# COMPARAÇÃO ENTRE O DF ANTERIOR E O QUE FOI ADICIONADO PARA A BASELINE
#------------------------------------------------------------------------
comparar_datasets(score_01_02,df_book_v3, 'FPD')

 ANÁLISE COMPARATIVA: FPD
 DATAFRAME 1 (Referência):
   - Shape: (2633900, 13)
   - Proporção de FPD: 21.23%

 DATAFRAME 2 (Atual):
   - Shape: (2633900, 86)
   - Proporção de FPD: 21.23%

 IMPACTO DA TRANSFORMAÇÃO:
   - Linhas removidas: 0 (0.00% de perda)
   - Variação no Target: 0.0000 p.p. (pontos percentuais)


In [ ]:
df_book_v3.select('var26_a_var77_MISSING').describe().show()

+-------+---------------------+
|summary|var26_a_var77_MISSING|
+-------+---------------------+
|  count|              2633900|
|   mean|  -0.5025802042598428|
| stddev|   0.5008886578066665|
|    min|                   -1|
|    max|                    1|
+-------+---------------------+



In [ ]:
#-----------------------------------------------------------------------------------------
#CRIANDO FLAG DE IDENTIFICAÇÃO PARA MISSING DO JOIN E ARRUMANDO FLAG DOS AUSENTE DA BASE
#----------------------------------------------------------------------------------------
df_telco = df_book_v3.withColumns({
    'var26_a_var77_MISSING' : F.when(F.col('var26_a_var77_MISSING') == -1, 0).otherwise(F.col('var26_a_var77_MISSING')),
    'var26_a_var93_NAO_SE_APLICA': F.when(F.col('var_92') == -1, 1).otherwise(0)
})

In [ ]:
df_telco.show(5)

+-----------------+--------------+-----------+------+--------+-------------+-------+--------+---+--------+----------------+--------+----------------+----------------+---+---+---+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------+---------------------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|Flag_OOT|MIG_Aquisicao|MIG_PRE|MIG_FLEX|FPD|SCORE_01|SCORE_01_MISSING|SCORE_02|SCORE_02_MISSING|MIG_SEM_MIGRACAO|CMV|NET|DTH|var_26|var_27|var_28|var_29|var_30|var_31|var_32|var_33|var_34|var_35|var_36|var_37|var_38|var_39|var_40|var_41|var_4

## Salvamento

In [ ]:
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Gold/plus_telco"
df_telco.write.mode("overwrite").parquet(path_silver)

## Relatório

Integração de variáveis de consumo e perfil de produtos da base Telco, fundamentada em uma abordagem de "Black-Box Feature Engineering" devido à anonimização dos atributos.

1 - **Natureza dos Dados:**

A base Telco é composta predominantemente por variáveis numéricas anonimizadas. Estes atributos representam o histórico quantitativo de utilização dos serviços pelo usuário. Na ausência de metadados descritivos para cada variável, o tratamento adotado foca na preservação da variância e na integridade estatística, permitindo que o modelo identifique padrões preditivos de forma agnóstica.

2 - **Atributos Integrados à ABT:**

* Identificação de Portfólio (`CMV`, `NET`, `DTH`): Variáveis indicadoras que segmentam  os produtos utilizados pelo cliente.

* Variáveis de Uso (`var_26` a `var_93`): Conjunto de atributos numéricos anonimizados que mensuram a intensidade e frequência de consumo.

* `var26_a_var77_MISSING:` Flag indicadora de ausência de dados na base de origem. Identifica registros onde a informação não foi capturada pelo sistema gerador.

* `var26_a_var93_NAO_SE_APLICA:` Flag resultante do processo de Left Join. Identifica clientes que não possuem histórico de uso registrado para estes produtos específicos, diferenciando-os dos clientes com dados ausentes na origem.

3 - **Estratégia de Imputação e Valores Sentinelas**

Para assegurar que o modelo de aprendizado de máquina interprete corretamente os diferentes estados de ausência de informação, foram definidos valores sentinelas distintos:

* Valor `-4`: Sentinela que identifica valores ausentes na base original. Representa uma falha de preenchimento ou indisponibilidade do dado no bureau.

* Valor `-1`: Sentinela que identifica valores ausentes pós-join. Indica que o CPF em questão não possui transacionalidade ou relacionamento histórico mapeado nesta tabela específica ("Não se aplica").

#

# 7 - Cadastral

In [ ]:
df = dfs['tabela_cadastral']

In [ ]:
df.show(5)

+-----------------+--------------+-----------+------+----+---------------+------------+----+--------+------------+----------------+------------------------+---------+---------------+-----------------+-------------------------+------------+------------------+--------------------------+-------------+-------------------+---------------------------+----------------+-----------------------+-------------------------------+----------+-----------------+-------------------------+--------------------+-------------------------+---------------------------------+------------+-------------------+-----------------+-------------------------+--------------------+-------------+----------------+------------------------+------+--------------+------+--------------+------+--------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2| FPD|STATUSRF|NUM_STATUSRF|DATADENASCIMENTO|DATADENASCIMENTO_MISSING|FUNC_PUBL|CARGO_FUNC_PUBL|SALARIO_FUNC_PUBL|SALARIO_FUNC_PUBL_MISSI

In [ ]:
Freq(df, 'STATUSRF')

/usr/local/lib/python3.12/dist-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


Quantidade de dominios 7
+-------------------------+------------+--------------+
|STATUSRF                 |qtd_absoluto|qtd_percentual|
+-------------------------+------------+--------------+
|REGULAR                  |3848697     |98.67         |
|PENDENTE DE REGULARIZACAO|31217       |0.8           |
|SEM_STATUS               |15154       |0.39          |
|SUSPENSA                 |2689        |0.07          |
|TITULAR FALECIDO         |2398        |0.06          |
|CANCELADA                |222         |0.01          |
|NULA                     |1           |0.0           |
+-------------------------+------------+--------------+

volumetria total: 3900378


'Freq da coluna STATUSRF'

In [ ]:
#------------------------------------
# APLICANDO OHE NA VARIAVEL STATUSRF
#------------------------------------

df = df.withColumns({
    'STATUSRF_REGULAR': F.when(F.col('STATUSRF') == 'REGULAR', 1).otherwise(0),
    'STATUSRF_PENDENTE_DE_REGULARIZACAO': F.when(F.col('STATUSRF') == 'PENDENTE DE REGULARIZACAO', 1).otherwise(0),
    'STATUSRF_SEM_STATUS': F.when(F.col('STATUSRF') == 'SEM STATUS', 1).otherwise(0),
    'STATUSRF_SUSPENSA': F.when(F.col('STATUSRF') == 'SUSPENSA', 1).otherwise(0),
    'STATUSRF_TITULAR_FALECIDO': F.when(F.col('STATUSRF') == 'TITULAR FALECIDO', 1).otherwise(0),
    'STATUSRF_CANCELADA': F.when(F.col('STATUSRF') == 'CANCELADA', 1).otherwise(0),
}).drop('STATUSRF')

In [ ]:
df.select(F.min('DATADENASCIMENTO')).show()

+---------------------+
|min(DATADENASCIMENTO)|
+---------------------+
|           1000-01-01|
+---------------------+



In [ ]:
#---------------------------------------------------------------------------------
# CALCULANDO IDADE DA DATA DE NASCIMENTO ATÉ A SAFRA (TRATAR OUTLIERS NA 2º FASE)
# --------------------------------------------------------------------------------
coluna_data_safra = F.to_date(F.concat(F.col('SAFRA'), F.lit('01')), 'yyyyMMdd')

df = df.withColumn(
    'IDADE',
    F.floor(
        F.months_between(coluna_data_safra, F.col('DATADENASCIMENTO')) / 12
    ).cast('int')
)

df = df.withColumns({
    'IDADE': F.when(F.col('DATADENASCIMENTO_MISSING') == 1, -4).otherwise(F.col('IDADE')),
    'IDADE_MISSING': F.when(F.col('DATADENASCIMENTO_MISSING') == 1, 1).otherwise(0)
}).drop('DATADENASCIMENTO', 'DATADENASCIMENTO_MISSING')

In [ ]:
df.show(5)

+-----------------+--------------+-----------+------+----+---------------+------------+----+------------+---------+---------------+-----------------+-------------------------+------------+------------------+--------------------------+-------------+-------------------+---------------------------+----------------+-----------------------+-------------------------------+----------+-----------------+-------------------------+--------------------+-------------------------+---------------------------------+------------+-------------------+-----------------+-------------------------+--------------------+-------------+----------------+------------------------+------+--------------+------+--------------+------+--------------+----------------+----------------------------------+-------------------+-------------------------+--------------------------+------------------+-----+-------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2| FPD|NUM_STATUSRF|FUNC_

In [ ]:
df.select('IDADE', 'IDADE_MISSING').describe().show()

+-------+------------------+--------------------+
|summary|             IDADE|       IDADE_MISSING|
+-------+------------------+--------------------+
|  count|           3900378|             3900378|
|   mean|   41.853975178816|0.004315222780971485|
| stddev|15.114051769213512| 0.06554847622110105|
|    min|                -4|                   0|
|    max|               131|                   1|
+-------+------------------+--------------------+



In [ ]:
Freq(df, 'CARGO_FUNC_PUBL', 10)

Quantidade de dominios 3321
+--------------------------------+------------+--------------+
|CARGO_FUNC_PUBL                 |qtd_absoluto|qtd_percentual|
+--------------------------------+------------+--------------+
|NAO_APLICAVEL                   |3838163     |98.4          |
|SOLDADORECRUTA                  |5820        |0.15          |
|SOLDADO                         |2722        |0.07          |
|PROFESSOR DO MAGISTERIO SUPERIOR|1678        |0.04          |
|CABO ENGAJADO                   |1495        |0.04          |
|TERCEIROSARGENTO                |1452        |0.04          |
|PROFESSOR EDUCACAO BASICA II    |1372        |0.04          |
|PROFESSOR                       |1359        |0.03          |
|PROFESSOR EDUCACAO BASICA I     |1067        |0.03          |
|PROFESSOR DE EDUCACAO BASICA    |997         |0.03          |
+--------------------------------+------------+--------------+
only showing top 10 rows


'Dominio muito granular'

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------
# CARGO FUNCIONARIO PUBLICO POSSUI UMA GRANULARIDADE MUITO ALTA TRATAR NA 2º FASE, VARIAVEL PARA BASELINE SERA EXCLUIDA
# ---------------------------------------------------------------------------------------------------------------------
df = df.drop('CARGO_FUNC_PUBL')

In [ ]:
# ----------------------------------------------------------------------------------------------------
# UF_BOLSA_FAMILIA EMBORA NÃO POSSUA UMA GRANULARIDADE MUITO ALTA PODE SER MELHOR TRATADA NA 2º FASE,
# EXCLUIDA  PARA BASELINE
# ----------------------------------------------------------------------------------------------------

df = df.drop('UF_BOLSA_FAMILIA')

In [ ]:
# ---------------------------------------------------------------------------------------------
# POR SE TRATAR DE UMA SAFRA/DATA PURA SERA RETIRADO DO MODELO BASELINE PARA EVITAR TENDENCIA,
# E SERA MELHOR TRATADO NA 2º FASE
#----------------------------------------------------------------------------------------------

df = df.drop('SAFRA_BOLSA_FAMILIA')

In [ ]:
Freq(df, 'STATUS_FUNC_PRIVADO')

Quantidade de dominios 3
+-------------------+------------+--------------+
|STATUS_FUNC_PRIVADO|qtd_absoluto|qtd_percentual|
+-------------------+------------+--------------+
|ADMITIDO           |1823762     |46.76         |
|NAO_APLICAVEL      |1490335     |38.21         |
|DISPENSADO         |586281      |15.03         |
+-------------------+------------+--------------+

volumetria total: 3900378


'Freq da coluna STATUS_FUNC_PRIVADO'

In [ ]:
#---------------------------------------------
# APLICANDO OHE NA COLUNA STATUS_FUNC_PRIVADO
#---------------------------------------------

df = df.withColumns({
    'STATUS_FUNC_PRIVADO_ADMITIDO': F.when(F.col('STATUS_FUNC_PRIVADO') == 'ADMITIDO', 1).otherwise(0),
    'STATUS_FUNC_PRIVADO_NAO_APLICAVEL': F.when(F.col('STATUS_FUNC_PRIVADO') == 'NAO_APLICAVEL', 1).otherwise(0),
    'STATUS_FUNC_PRIVADO_DISPENSADO': F.when(F.col('STATUS_FUNC_PRIVADO') == 'DISPENSADO', 1).otherwise(0),
}).drop('STATUS_FUNC_PRIVADO')

In [ ]:
# -------------------------------------------------------------------------------
# DATA_FUNC_PRIVADO POR SE TRATAR DE UMA DATA PURA SERA RETIRADO DO BASELINE PARA
# EVITAR TENDENCIA, E SERA MELHOR TRATADO NA 2º FASE
# -------------------------------------------------------------------------------
df = df.drop('DATA_FUNC_PRIVADO')

In [ ]:
Freq(df, 'CONSOLIDADO')

Quantidade de dominios 64
+---------------------------------------------------------------------+------------+--------------+
|CONSOLIDADO                                                          |qtd_absoluto|qtd_percentual|
+---------------------------------------------------------------------+------------+--------------+
|FUNC_PRIVADO                                                         |945989      |24.25         |
|AUX_EMRG FUNC_PRIVADO                                                |535967      |13.74         |
|SEM_INFORMACAO                                                       |467469      |11.99         |
|AUX_EMRG                                                             |371795      |9.53          |
|APOSENTADO FUNC_PRIVADO                                              |268762      |6.89          |
|AUX_EMRG BOLSA_FAMILIA                                               |226240      |5.8           |
|APOSENTADO                                                           |198

'Freq da coluna CONSOLIDADO'

In [ ]:
#-----------------------------------------------------------------------
# VARIAVEL CONSOLIDADO É MUITO GRANULAR SERA EXCLUIDA PARA A BASELINE E
# MELHOR TRATADA PARA A 2º FASE
#-----------------------------------------------------------------------

df = df.drop('CONSOLIDADO')

In [ ]:
# ---------------------------------------------------------------------------
# PADRONIZANDO VARIAVEIS: 'VALOR_EMPR_DIRETOR', 'NUMERO_APOSENTADO', 'var_02'
#----------------------------------------------------------------------------

variaveis = ['VALOR_EMPR_DIRETOR', 'NUMERO_APOSENTADO', 'var_02']

for var in variaveis:
  df = df.withColumn(
      var,
      F.when(F.col(var) == -999, -1).otherwise(F.col(var))
  )

In [ ]:
# ------------------------------------------
# PADRONIZANDO A VARIAVEL var_07_MONETARIO
#-------------------------------------------
df = df.withColumn(
    'var_07_MONETARIO',
    F.round(
        F.when(F.col('var_07_MONETARIO') == -999, -1)
         .otherwise(F.col('var_07_MONETARIO')),
        2
    )
)

In [ ]:
#------------------------------------------------------------------------
# VARIAVEL CEP_3_digitos É MUITO GRANULAR SERA EXCLUIDA PARA A BASELINE E
# MELHOR TRATADA PARA A 2º FASE
#------------------------------------------------------------------------

df = df.drop('CEP_3_digitos')

In [ ]:
df.show(5)

+-----------------+--------------+-----------+------+----+---------------+------------+----+------------+---------+-----------------+-------------------------+------------+------------------+--------------------------+-------------+---------------------------+-----------------------+-------------------------------+----------+-----------------+-------------------------+--------------------+-------------------------+---------------------------------+------------+-------------------------+----------------+------------------------+------+--------------+------+--------------+------+--------------+----------------+----------------------------------+-------------------+-------------------------+--------------------------+------------------+-----+-------------+----------------------------+---------------------------------+------------------------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|PROD|FLAG_INSTALACAO|   flag_mig2| FPD|NUM_STATUSRF|FUNC_PUBL|SALARIO_FUNC_PUBL|SALARIO_F

In [ ]:
# ------------------------------------------
# PADRONIZANDO A VARIAVEL GRUPO_CONTROLE
#-------------------------------------------
df = df.withColumn(
    'GRUPO_CONTROLE',
    F.col('GRUPO_CONTROLE').cast('int'),
)

In [ ]:
#-------------------------------------------------
# REMOVENDO CLIENTES QUE NÃO CONTRATARAM (SEM FPD)
#-------------------------------------------------
df = df.dropna(subset = ['FPD'])

In [ ]:
df.show(5)

+-----------------+--------------+-----------+------+----+---------------+---------+---+------------+---------+-----------------+-------------------------+------------+------------------+--------------------------+-------------+---------------------------+-----------------------+-------------------------------+----------+-----------------+-------------------------+--------------------+-------------------------+---------------------------------+------------+-------------------------+----------------+------------------------+------+--------------+------+--------------+------+--------------+----------------+----------------------------------+-------------------+-------------------------+--------------------------+------------------+-----+-------------+----------------------------+---------------------------------+------------------------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|PROD|FLAG_INSTALACAO|flag_mig2|FPD|NUM_STATUSRF|FUNC_PUBL|SALARIO_FUNC_PUBL|SALARIO_FUNC_PUBL

In [ ]:
df = df.drop('PROD','FLAG_INSTALACAO','flag_mig2', 'SAFRA_BOLSA_FAMILIA_MISSING', 'DATA_FUNC_PRIVADO_MISSING')

In [ ]:
# import da gold
caminho_gold = "/content/gdrive/MyDrive/Hackathon_POD/Gold/plus_telco"
try:
    df_gold = spark.read.parquet(f'{caminho_gold}/*.parquet')
    print("Tabela Gold carregada com sucesso!")
    print(f"Total de linhas: {df_gold.count()}")
except Exception as e:
    print(f" Erro ao carregar: {e}")
    print("Verifique se o caminho está correto.")

Tabela Gold carregada com sucesso!
Total de linhas: 2633900


In [ ]:
get_shape(df)

('Quanitade de linhas: 2696621', 'Quanitade de Colunas: 41')

In [ ]:
get_shape(df_gold)

('Quanitade de linhas: 2633900', 'Quanitade de Colunas: 87')

In [ ]:
#----------------------------------------
#INNER JOIN ENTRE CA
#
cols_gold = df_gold.columns
cols_add = df.columns

cols_novas = [c for c in cols_add if c not in cols_gold or c == 'ID_UNICO']

df_book_final = df_gold.join(
    df.select(cols_novas),
    on='ID_UNICO',
    how='inner'
)

print(f"Linhas na Gold: {df_gold.count()}")
print(f"Linhas na Cadastral: {df.count()}")
print(f"Linhas após Inner Join: {df_book_final.count()}")

Linhas na Gold: 2633900
Linhas na Cadastral: 2696621
Linhas após Inner Join: 2633900


In [ ]:
df_book_final.show(5)

+-----------------+--------------+-----------+------+--------+-------------+-------+--------+---+--------+----------------+--------+----------------+----------------+---+---+---+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------+---------------------------+------------+---------+-----------------+-------------------------+------------+------------------+--------------------------+-------------+-----------------------+-------------------------------+----------+-----------------+-------------------------+--------------------+------------------------

In [ ]:
comparar_datasets(df, df_book_final, 'FPD')

 ANÁLISE COMPARATIVA: FPD
 DATAFRAME 1 (Referência):
   - Shape: (2696621, 41)
   - Proporção de FPD: 21.27%

 DATAFRAME 2 (Atual):
   - Shape: (2633900, 123)
   - Proporção de FPD: 21.23%

 IMPACTO DA TRANSFORMAÇÃO:
   - Linhas removidas: 62721 (2.33% de perda)
   - Variação no Target: 0.0402 p.p. (pontos percentuais)


In [ ]:
comparar_datasets(df_gold, df_book_final, 'FPD')

 ANÁLISE COMPARATIVA: FPD
 DATAFRAME 1 (Referência):
   - Shape: (2633900, 87)
   - Proporção de FPD: 21.23%

 DATAFRAME 2 (Atual):
   - Shape: (2633900, 123)
   - Proporção de FPD: 21.23%

 IMPACTO DA TRANSFORMAÇÃO:
   - Linhas removidas: 0 (0.00% de perda)
   - Variação no Target: 0.0000 p.p. (pontos percentuais)


## Salvamento

In [ ]:
path_silver = "/content/gdrive/MyDrive/Hackathon_POD/Gold/plus_cadastral"
df_book_final.write.mode("overwrite").parquet(path_silver)

In [ ]:
df.show(5)

+-----------------+--------------+-----------+------+---+------------+---------+-----------------+-------------------------+------------+------------------+--------------------------+-------------+-----------------------+-------------------------------+----------+-----------------+-------------------------+--------------------+-------------------------+---------------------------------+------------+----------------+------------------------+------+--------------+------+--------------+------+--------------+----------------+----------------------------------+-------------------+-----------------+-------------------------+------------------+-----+-------------+----------------------------+---------------------------------+------------------------------+
|         ID_UNICO|GRUPO_CONTROLE|    NUM_CPF| SAFRA|FPD|NUM_STATUSRF|FUNC_PUBL|SALARIO_FUNC_PUBL|SALARIO_FUNC_PUBL_MISSING|EMPR_DIRETOR|VALOR_EMPR_DIRETOR|VALOR_EMPR_DIRETOR_MISSING|BOLSA_FAMILIA|BENEFICIO_BOLSA_FAMILIA|BENEFICIO_BOLSA_FAM

In [ ]:
df_book_final.show(5)

+-----------------+--------------+-----------+------+--------+-------------+-------+--------+---+--------+----------------+--------+----------------+----------------+---+---+---+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------+---------------------------+------------+---------+-----------------+-------------------------+------------+------------------+--------------------------+-------------+---------------------------+-----------------------+-------------------------------+----------+-----------------+-------------------------+-----------------

## Relatório
Descrição da integração de atributos cadastrais e socioeconômicos à ABT Baseline. A base cadastral fornece uma camada fundamental de perfilamento, embora sua densidade varie conforme o canal de aquisição e o tipo de produto do cliente.

1 - **Metodologia de Integração**

Para a construção da Baseline, optou-se pela execução de um Inner Join entre a base Gold e a base Cadastral.

- Impacto na Volumetria: Observou-se uma perda residual de 2,33% nos registros. Tal redução é considerada estatisticamente aceitável nesta fase do projeto, dado o ganho substancial de informações granulares que serão incorporadas ao modelo.

- Critério de Junção: A unificação assegura que todos os proponentes na ABT possuam o perfil socioeconômico básico preenchido para o treinamento inicial.

2 - **Dicionário de Variáveis e Atributos**

As variáveis foram segmentadas em blocos funcionais para melhor interpretação do modelo:

**A. Perfil de Renda e Ocupação**

- `FUNC_PUBL` / `SALARIO_FUNC_PUBL`: Identifica servidores públicos e sua respectiva faixa salarial.

  * Nota: O valor -1 é utilizado para diferenciar clientes que não são servidores públicos de servidores que possuem valor registrado como zero.

- `EMPR_DIRETOR` / `VALOR_EMPR_DIRETOR`: Indicadores de cargos de alta gestão e empresários, com métrica de relevância associada (range 1 a 154).

- `FUNC_PRIVADO` / `STATUS_FUNC_PRIVADO`: Identifica funcionários do setor privado e seu status atual (Admitido, Dispensado ou Não Aplicável).

- `var_07_MONETARIO`: Atributo numérico de alta magnitude relacionado ao poder de compra ou patrimônio.

**B. Benefícios e Auxílios Governamentais**

- `BOLSA_FAMILIA` / `BENEFICIO_BOLSA_FAMILIA`: Identifica beneficiários atuais ou pretéritos do programa e o valor do benefício.

- `AUXILLIO_EMERGENCIAL` / `MESES_AUXILIO_EMERGENCIAL`: Histórico de recebimento de auxílio emergencial e tempo de permanência no programa.

**C. Situação Demográfica e Previdenciária**

- `IDADE`: Idade cronológica em anos calculada a partir da data de nascimento até a data de referência da Safra.

- `APOSENTADO` / `NUMERO_APOSENTADO`: Identifica a condição de aposentadoria e o registro de natureza administrativa associado.

- `STATUSRF_...` (Flags de Receita Federal): Conjunto de variáveis binárias que descrevem a situação cadastral do CPF (Regular, Pendente, Suspensa, Falecido ou Cancelada).

**D. Variáveis Anonimizadas de Bureau**
- `var_02`, `var_03`, `var_05`: Atributos numéricos de natureza técnica fornecidos pelo bureau, mantidos para extração de padrões não óbvios pelo modelo.

3 -  **Governança de Dados e Tratamento de Omissos**

Para todas as variáveis acima, foi implementada uma estrutura de Flags de Missing (ex: `IDADE_MISSING`, `var_03_MISSING`).

- Tratamento de Nulos: Valores -1 foram adotados como sentinelas para informações nulas na base original.

- Lógica de Inferência: A ausência dessas informações é interpretada como "Não se Aplica". Por exemplo, um valor nulo em MESES_AUXILIO_EMERGENCIAL convertido para -1 indica que o cliente não pertence ao grupo de beneficiários, transformando a falta de dado em um atributo informativo.

#

# 8 - Tabelas DIM

In [ ]:
tabelas_dim = [
    'tabela_bi_bi_dim_status_plataforma',
    'tabela_bi_dim_canal_aquisicao_credito',
    'tabela_bi_dim_forma_pagamento',
    'tabela_bi_dim_instituicao',
    'tabela_bi_dim_plano_preco',
    'tabela_bi_dim_plataforma',
    'tabela_bi_dim_promocao_credito',
    'tabela_bi_dim_tecnologia',
    'tabela_bi_dim_tipo_credito',
    'tabela_bi_dim_tipo_insercao',
    'tabela_bi_dim_tipo_recarga']

In [ ]:
for i in tabelas_dim:
  print(i)
  dfs[i].show(40)
  print('\n')

tabela_bi_bi_dim_status_plataforma
+---------------------+---------------------+---------+------------------+--------------+-------------------+----------------------+
|COD_STATUS_PLATAFORMA|DSC_STATUS_PLATAFORMA|IND_ATIVO|DAT_ATUALIZACAO_DW|DAT_CRIACAO_DW|COD_STATUS_PLAT_GRP|IND_STS_PLAT_GRP_ATIVO|
+---------------------+---------------------+---------+------------------+--------------+-------------------+----------------------+
|                    A|                Ativo|        S|        2006-10-16|    2006-10-16|                  A|                     S|
|                  ZB1|           Expirado 1|        S|        2006-10-16|    2006-10-16|                ZB1|                     S|
|                  PRE|            Pré-Ativo|        N|        2006-10-16|    2006-10-16|                PRE|                     N|
|                  ZB2|           Expirado 2|        N|        2006-10-16|    2006-10-16|                ZB2|                     N|
|                    C|         De

#

# 9 - Recarga

In [14]:
# import da gold
caminho_gold = "/content/gdrive/MyDrive/Hackathon_POD/Gold/plus_cadastral"
try:
    df_gold = spark.read.parquet(f'{caminho_gold}/*.parquet')
    print("Tabela Gold carregada com sucesso!")
    print(f"Total de linhas: {df_gold.count()}")
except Exception as e:
    print(f" Erro ao carregar: {e}")
    print("Verifique se o caminho está correto.")

Tabela Gold carregada com sucesso!
Total de linhas: 2633900


In [15]:
df = dfs['tabela_recarga']

In [ ]:
df.show()

+-----------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+--------------+
|    NUM_CPF|DW_NUM_NTC|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|DW_NUM_CLIENTE|COD_TECNOLOGIA_DW|COD_CANAL_AQUISICAO|COD_TIPO_CREDITO|COD_PROMOCAO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAl|COD_PLATAFORMA_ATU|COD_STATUS_PLATAFORMA|IND_METODO_PAGAMENTO|DW_PLANO_TARIFACAO|DW_TIPO_RECARGA|DW_TIPO_INSERCAO|DW_FORMA_PAGAMENTO|DW_INSTITUICAO|COD_GRUPO_CARTAO|DSC_GRUPO_CARTAO_WPP|FLAG_SOS|VALOR_SOS|GRUPO_CONTROLE|
+-----------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+

In [16]:
df_skeleton = df_gold.select('ID_UNICO', 'NUM_CPF', 'SAFRA')

In [17]:
#------------------------------------------------------------------
# Converter a SAFRA (String '202410') para Data (Date '2024-10-01')
# -----------------------------------------------------------------

df_skeleton = df_skeleton.withColumn(
    'DATA_SAFRA',
    F.to_date(F.concat(F.col('SAFRA'), F.lit('01')), 'yyyyMMdd')
)

In [18]:
# -------------------------------------------------
# Join da Recarga com o Esqueleto pelo CPF
# Só recarga de quem está no book quem não descarta
#---------------------------------------------------

df_join_temporal = df_skeleton.join(df, on='NUM_CPF', how='inner')

In [19]:
# -----------------------------
# CRIANDO TABELA INTERMEDIARIA
#------------------------------


df_recarga_valida = df_join_temporal.filter(
    F.col('DAT_INSERCAO_CREDITO') < F.col('DATA_SAFRA')
)

In [20]:
df_recarga_valida.show(5)

+-----------+-----------------+------+----------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+--------------+
|    NUM_CPF|         ID_UNICO| SAFRA|DATA_SAFRA|DW_NUM_NTC|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|DW_NUM_CLIENTE|COD_TECNOLOGIA_DW|COD_CANAL_AQUISICAO|COD_TIPO_CREDITO|COD_PROMOCAO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAl|COD_PLATAFORMA_ATU|COD_STATUS_PLATAFORMA|IND_METODO_PAGAMENTO|DW_PLANO_TARIFACAO|DW_TIPO_RECARGA|DW_TIPO_INSERCAO|DW_FORMA_PAGAMENTO|DW_INSTITUICAO|COD_GRUPO_CARTAO|DSC_GRUPO_CARTAO_WPP|FLAG_SOS|VALOR_SOS|GRUPO_CONTROLE|
+-----------+-----------------+------+----------+----------+--------------------+--------------------+------

In [ ]:
get_shape(df_recarga_valida)

('Quanitade de linhas: 55557955', 'Quanitade de Colunas: 28')

In [21]:
# -------------------------
# CRIANDO ESTRUTURA DO BOOK
# -------------------------


# Agrupando por ID_UNICO
df_book_recarga = df_recarga_valida.groupBy('ID_UNICO').agg(
    # 1. Total recarregado
    F.sum('VAL_CREDITO_INSERIDO').alias('REC_TOTAL_VALOR_HIST'),

    # 2. Quantidade de vezes que recarregou
    F.count('VAL_CREDITO_INSERIDO').alias('REC_QTD_RECARGAS_HIST'),

    # 3. Média do valor de recarga
    F.avg('VAL_CREDITO_INSERIDO').alias('REC_MEDIA_VALOR_HIST'),

    # 4. Maior recarga já feita
    F.max('VAL_CREDITO_INSERIDO').alias('REC_MAX_VALOR_HIST'),

    # 5. CORREÇÃO AQUI: Usar a coluna de DATA, não a de VALOR
    F.max('DAT_INSERCAO_CREDITO').alias('DATA_ULTIMA_RECARGA')
)

# Cálculo final da recência (Dias desde a última recarga)
df_book_recarga = df_book_recarga.withColumn(
    'REC_DIAS_DESDE_ULTIMA',
    F.datediff(
        F.to_date(F.concat(F.substring(F.col('ID_UNICO'), -6, 6), F.lit('01')), 'yyyyMMdd'),
        F.col('DATA_ULTIMA_RECARGA')
    )
).drop('DATA_ULTIMA_RECARGA')

In [ ]:
df_book_recarga.show(5)

+-----------------+--------------------+---------------------+--------------------+------------------+---------------------+
|         ID_UNICO|REC_TOTAL_VALOR_HIST|REC_QTD_RECARGAS_HIST|REC_MEDIA_VALOR_HIST|REC_MAX_VALOR_HIST|REC_DIAS_DESDE_ULTIMA|
+-----------------+--------------------+---------------------+--------------------+------------------+---------------------+
|787UU79YXXT202503|               487.0|                   61|   7.983606557377049|              40.0|                    9|
|79NWUX9YNTW202502|                 0.0|                   17|                 0.0|               0.0|                   18|
|7TNTN7XZ88Y202412|                 0.0|                    7|                 0.0|               0.0|                   20|
|7TXZX9NW9N8202410|                45.0|                   30|                 1.5|              15.0|                   11|
|7TZ7TUZ7T87202412|                85.0|                   11|  7.7272727272727275|              25.0|                    4|


In [22]:
# ---------------------------------------------
# IMPORTANDO PLANOS DA tabela_bi_dim_plataforma
#----------------------------------------------

df_dimensao_clean = dfs['tabela_bi_dim_plataforma'].select(
    F.col("DSC_PLATAFORMA").alias("COD_PLATAFORMA_ATU"),
    F.col("DSC_GRUPO_PLATAFORMA")
)

df_recarga_enriquecida = df_recarga_valida.join(
    df_dimensao_clean,
    on="COD_PLATAFORMA_ATU",
    how="left"
)

df_recarga_enriquecida = df_recarga_enriquecida.withColumn(
    "CATEGORIA_FINAL",
    F.when(F.col("DSC_GRUPO_PLATAFORMA") == "Pré Pago", "PRE_PAGO")
     .when(F.col("DSC_GRUPO_PLATAFORMA") == "Controle", "CONTROLE")
     .when(F.col("DSC_GRUPO_PLATAFORMA") == "Pós Pago", "POS_PAGO")
     .otherwise("OUTROS")
)

In [23]:
# --------------------------
# ADICIONANDO PLANOS AO BOOK
#---------------------------

df_book_recarga = df_recarga_enriquecida.groupBy('ID_UNICO').agg(
    # --- Métricas Originais (Mantidas) ---
    F.sum('VAL_CREDITO_INSERIDO').alias('REC_TOTAL_VALOR_HIST'),
    F.count('VAL_CREDITO_INSERIDO').alias('REC_QTD_RECARGAS_HIST'),
    F.avg('VAL_CREDITO_INSERIDO').alias('REC_MEDIA_VALOR_HIST'),
    F.max('VAL_CREDITO_INSERIDO').alias('REC_MAX_VALOR_HIST'),
    F.max('DAT_INSERCAO_CREDITO').alias('DATA_ULTIMA_RECARGA'),

    # --- NOVAS COLUNAS: Perfil da Plataforma (One-Hot Encoding) ---
    # Conta quantas vezes apareceu como PRE_PAGO
    F.sum(F.when(F.col("CATEGORIA_FINAL") == "PRE_PAGO", 1).otherwise(0)).alias('QTD_PRE_PAGO'),

    # Conta quantas vezes apareceu como CONTROLE
    F.sum(F.when(F.col("CATEGORIA_FINAL") == "CONTROLE", 1).otherwise(0)).alias('QTD_CONTROLE'),

    # Conta quantas vezes apareceu como POS_PAGO
    F.sum(F.when(F.col("CATEGORIA_FINAL") == "POS_PAGO", 1).otherwise(0)).alias('QTD_POS_PAGO'),

    # Conta o resto (Outros)
    F.sum(F.when(F.col("CATEGORIA_FINAL") == "OUTROS", 1).otherwise(0)).alias('QTD_OUTROS')
)

# --- Cálculo de Recência Mantido ---
df_book_recarga = df_book_recarga.withColumn(
    'REC_DIAS_DESDE_ULTIMA',
    F.datediff(
        F.to_date(F.concat(F.substring(F.col('ID_UNICO'), -6, 6), F.lit('01')), 'yyyyMMdd'),
        F.col('DATA_ULTIMA_RECARGA')
    )
).drop('DATA_ULTIMA_RECARGA')

In [24]:
df_book_recarga.show(5)

+-----------------+--------------------+---------------------+--------------------+------------------+------------+------------+------------+----------+---------------------+
|         ID_UNICO|REC_TOTAL_VALOR_HIST|REC_QTD_RECARGAS_HIST|REC_MEDIA_VALOR_HIST|REC_MAX_VALOR_HIST|QTD_PRE_PAGO|QTD_CONTROLE|QTD_POS_PAGO|QTD_OUTROS|REC_DIAS_DESDE_ULTIMA|
+-----------------+--------------------+---------------------+--------------------+------------------+------------+------------+------------+----------+---------------------+
|787UU79YXXT202503|               487.0|                   61|   7.983606557377049|              40.0|          58|           3|           0|         0|                    9|
|79NWUX9YNTW202502|                 0.0|                   17|                 0.0|               0.0|           0|          17|           0|         0|                   18|
|7TNTN7XZ88Y202412|                 0.0|                    7|                 0.0|               0.0|           0|          

In [25]:
# ------------------------------------
# ADICIONANDO INFORMAÇÕES SOS AO BOOK
#-------------------------------------

df_book_recarga = df_recarga_enriquecida.groupBy('ID_UNICO').agg(
    # --- 1. Métricas Originais Mantidas ---
    F.sum('VAL_CREDITO_INSERIDO').alias('REC_TOTAL_VALOR_HIST'),
    F.count('VAL_CREDITO_INSERIDO').alias('REC_QTD_RECARGAS_HIST'),
    F.avg('VAL_CREDITO_INSERIDO').alias('REC_MEDIA_VALOR_HIST'),
    F.max('VAL_CREDITO_INSERIDO').alias('REC_MAX_VALOR_HIST'),
    F.max('DAT_INSERCAO_CREDITO').alias('DATA_ULTIMA_RECARGA'),

    # --- 2. Perfil da Plataforma Mantido ---
    F.sum(F.when(F.col("CATEGORIA_FINAL") == "PRE_PAGO", 1).otherwise(0)).alias('QTD_PRE_PAGO'),
    F.sum(F.when(F.col("CATEGORIA_FINAL") == "CONTROLE", 1).otherwise(0)).alias('QTD_CONTROLE'),
    F.sum(F.when(F.col("CATEGORIA_FINAL") == "POS_PAGO", 1).otherwise(0)).alias('QTD_POS_PAGO'),
    F.sum(F.when(F.col("CATEGORIA_FINAL") == "OUTROS", 1).otherwise(0)).alias('QTD_OUTROS'),

    # --- 3. NOVAS MÉTRICAS: SOS (Crédito de Emergência) ---
    F.sum(F.col('FLAG_SOS').cast('int')).alias('QTD_SOS'),

    # Soma total do valor emprestado
    F.sum('VALOR_SOS').alias('TOTAL_VALOR_SOS'),

    # Média do valor solicitado (quando houve solicitação)
    F.avg('VALOR_SOS').alias('MEDIA_SOS')
)

# --- Cálculo de Recência Mantido ---
df_book_recarga = df_book_recarga.withColumn(
    'REC_DIAS_DESDE_ULTIMA',
    F.datediff(
        F.to_date(F.concat(F.substring(F.col('ID_UNICO'), -6, 6), F.lit('01')), 'yyyyMMdd'),
        F.col('DATA_ULTIMA_RECARGA')
    )
).drop('DATA_ULTIMA_RECARGA')

In [26]:
df_book_recarga.show(5)

+-----------------+--------------------+---------------------+--------------------+------------------+------------+------------+------------+----------+-------+---------------+------------------+---------------------+
|         ID_UNICO|REC_TOTAL_VALOR_HIST|REC_QTD_RECARGAS_HIST|REC_MEDIA_VALOR_HIST|REC_MAX_VALOR_HIST|QTD_PRE_PAGO|QTD_CONTROLE|QTD_POS_PAGO|QTD_OUTROS|QTD_SOS|TOTAL_VALOR_SOS|         MEDIA_SOS|REC_DIAS_DESDE_ULTIMA|
+-----------------+--------------------+---------------------+--------------------+------------------+------------+------------+------------+----------+-------+---------------+------------------+---------------------+
|787UU79YXXT202503|               487.0|                   61|   7.983606557377049|              40.0|          58|           3|           0|         0|     10|           80.0|1.3114754098360655|                    9|
|79NWUX9YNTW202502|                 0.0|                   17|                 0.0|               0.0|           0|          17|

In [27]:
#---------------------
# ARREDONDANDO VALORES
#---------------------

cols_para_arredondar = [
    'REC_TOTAL_VALOR_HIST',
    'REC_MEDIA_VALOR_HIST',
    'MEDIA_SOS'
]
for coluna in cols_para_arredondar:
    df_book_recarga = df_book_recarga.withColumn(coluna, F.round(F.col(coluna), 2))

In [28]:
df_book_recarga.show(5)

+-----------------+--------------------+---------------------+--------------------+------------------+------------+------------+------------+----------+-------+---------------+---------+---------------------+
|         ID_UNICO|REC_TOTAL_VALOR_HIST|REC_QTD_RECARGAS_HIST|REC_MEDIA_VALOR_HIST|REC_MAX_VALOR_HIST|QTD_PRE_PAGO|QTD_CONTROLE|QTD_POS_PAGO|QTD_OUTROS|QTD_SOS|TOTAL_VALOR_SOS|MEDIA_SOS|REC_DIAS_DESDE_ULTIMA|
+-----------------+--------------------+---------------------+--------------------+------------------+------------+------------+------------+----------+-------+---------------+---------+---------------------+
|787UU79YXXT202503|               487.0|                   61|                7.98|              40.0|          58|           3|           0|         0|     10|           80.0|     1.31|                    9|
|79NWUX9YNTW202502|                 0.0|                   17|                 0.0|               0.0|           0|          17|           0|         0|      0|    

In [29]:
get_shape(df_gold)

('Quanitade de linhas: 2633900', 'Quanitade de Colunas: 123')

In [30]:
get_shape(df_book_recarga)

('Quanitade de linhas: 1891934', 'Quanitade de Colunas: 13')

In [31]:
#-----------------------
# REALIZANDO INNER JOIN
#-----------------------

df_abt_com_recarga = df_gold.join(
    df_book_recarga,
    on='ID_UNICO',
    how='inner'
)

In [32]:
comparar_datasets(df_gold, df_abt_com_recarga, 'FPD')

 ANÁLISE COMPARATIVA: FPD
 DATAFRAME 1 (Referência):
   - Shape: (2633900, 123)
   - Proporção de FPD: 21.23%

 DATAFRAME 2 (Atual):
   - Shape: (1891934, 135)
   - Proporção de FPD: 22.40%

 IMPACTO DA TRANSFORMAÇÃO:
   - Linhas removidas: 741966 (28.17% de perda)
   - Variação no Target: -1.1683 p.p. (pontos percentuais)


## Salvamento

In [ ]:
caminho_abt = '/content/gdrive/MyDrive/Hackathon_POD/Gold/plus_recarga.parquet'
df_abt_com_recarga.write.mode('overwrite').parquet(caminho_abt)

## Relatório

**1 - Metodologia de Construção**
- A construção deste book seguiu o princípio de Corte Temporal Reversivo para garantir a integridade do modelo e evitar o Data Leakage (vazamento de - dados).

- Ponto de Corte: Para cada registro, foram considerados apenas os eventos de recarga ocorridos em data estritamente inferior à data da SAFRA (dia 01 do mês de referência).

- Granularidade: Os dados transacionais originais (N linhas por CPF) foram agregados para a granularidade de ID_UNICO (1 linha por CPF/Safra).

- Tratamento de Plataformas: Utilizou-se uma tabela de dimensão para categorizar o COD_PLATAFORMA_ATU, permitindo identificar o comportamento de consumo por perfil de produto (Pré, Controle, Pós).

2 - **Dicionário de Variáveis e Relevância Estratégica**

| Variável | Lógica de Construção | Relevância para o Risco de Crédito
| :--- | :--- | :--- |
| **`REC_TOTAL_VALOR_HIST`** | Soma de todos os valores de recarga (`VAL_CREDITO_INSERIDO`) no período válido. |Indica o LTV (Lifetime Value) e a capacidade financeira histórica do cliente.
| **`REC_QTD_RECARGAS_HIST`** | Contagem total de eventos de recarga. | Mede a estabilidade. Clientes com recargas frequentes podem demonstram maior engajamento e previsibilidade.
| **`REC_MEDIA_VALOR_HIST`** | Média aritmética dos valores inseridos, arredondada para 2 casas decimais. | Define o ticket médio. Ajuda a segmentar o poder aquisitivo.
| **`REC_MAX_VALOR_HISTT`** | Valor máximo já recarregado em uma única transação. | Indica o teto de desembolso esporádico do cliente.
| **`REC_DIAS_DESDE_ULTIMA`** | Diferença em dias entre a data da Safra e a `DATA_ULTIMA_RECARGA`. | Recência: Possibilidade de ser um preditor forte. Com a hipótese de quanto maior o tempo sem atividade, maior o risco inadimplência.
| **`QTD_PRE_PAGO` / `CONTROLE` / `POS_PAGO`** | Soma binária (Flag 0/1) baseada no de-para da tabela de dimensão de plataformas. | Além de identificar o Plano, permite diferenciar "consumo zero" por falta de uso de "consumo zero" por característica do plano (ex: Controle/Pós).
| **`QTD_SOS`** | Soma da FLAG_SOS convertida em inteiro. | Indica a frequência de uso de Crédito de Emergência. O uso excessivo pode sinalizar fragilidade financeira momentânea.
| **`TOTAL_VALOR_SOS`** | Soma acumulada dos valores de crédito emergencial contratados. | Quantifica a dependência do cliente em relação a microempréstimos da operadora.
| **`MEDIA_SOS`** | Valor médio das solicitações de SOS, ignorando períodos sem uso. | Identifica o comportamento padrão em situações de falta de saldo.


**3 - Considerações Técnicas sobre a Integração (Inner Join)**

A opção pelo Inner Join entre a df_gold e o df_book_recarga foi estratégica para esta fase de Baseline. Esta decisão garante que:

* 1 - Qualidade da Informação: O modelo será treinado apenas com indivíduos que possuem histórico de atividade mensurável, reduzindo o ruído causado por CPFs inativos ou sem informações de consumo.

* 2 -Saneamento de Tipos: Todas as variáveis monetárias e de média foram padronizadas para duas casas decimais, assegurando que o algoritmo não interprete variações infinitesimais de ponto flutuante como informação relevante.

* 3 - Variáveis Sentinela: Manteve-se a coerência com as flags de missing tratadas anteriormente, permitindo que o modelo diferencie a ausência de informação do valor zero real.

#

# 10 - Pagamento





In [34]:
df_gold = df_abt_com_recarga

In [37]:
df_gold.show(5)

+-----------------+--------------+-----------+------+--------+-------------+-------+--------+---+--------+----------------+--------+----------------+----------------+---+---+---+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------+---------------------------+------------+---------+-----------------+-------------------------+------------+------------------+--------------------------+-------------+-----------------------+-------------------------------+----------+-----------------+-------------------------+--------------------+------------------------

In [35]:
df = dfs['tabela_pagamento']

In [36]:
df.show(5)

+-----------+------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+--------------------

In [38]:
#--------------------
# NORMALIZANDO DATAS
#--------------------

df  = df.withColumns({
    'DAT_STATUS_FATURA': F.to_date(F.col('DAT_STATUS_FATURA'), "ddMMMyyyy:HH:mm:ss"),
    'DAT_CRIACAO_DW' : F.to_date(F.col('DAT_CRIACAO_DW'), "ddMMMyyyy:HH:mm:ss"),

})

In [39]:
# ------------------------------
# PREPARANDO ESQUELETO
# ------------------------------
df_skeleton = df_gold.select('ID_UNICO', 'NUM_CPF', 'SAFRA') \
    .withColumn('DATA_SAFRA', F.to_date(F.concat(F.col('SAFRA'), F.lit('01')), 'yyyyMMdd'))

In [40]:
# -----------------------------
# INNER JOIN COM BASE PAGAMENTO
# ------------------------------

df_join_pagamento = df_skeleton.join(df, on='NUM_CPF', how='inner')

In [41]:
# ----------------
# FILTRO TEMPORAL
# ----------------

df_pagamento_filtrado = df_join_pagamento.filter(
    F.col('DAT_STATUS_FATURA') < F.col('DATA_SAFRA')
)

In [42]:
# -----------------------------
# CRIANDO TABELA INTERMEDIARIA
#------------------------------

df_pagamento_valido = df_pagamento_filtrado.withColumn(
    'DIAS_ATRASO_CALC',
    F.when(
        (F.col('DAT_STATUS_PAGAMENTO') > '1900-01-01') &
        (F.col('DAT_VENCIMENTO_CREDITO') < '9999-12-31'),
        F.datediff(F.col('DAT_STATUS_PAGAMENTO'), F.col('DAT_VENCIMENTO_CREDITO'))
    ).otherwise(0)
).withColumn(
    'FLAG_ATRASO',
    F.when(F.col('DIAS_ATRASO_CALC') > 0, 1).otherwise(0)
)


In [43]:
# -----------------------------
# CRIANDO BOOK DE PAGAMENTO
#------------------------------

df_book_pagamento = df_pagamento_valido.groupBy('ID_UNICO').agg(
    # --- Métricas de Volume ---
    F.count('SEQ_FATURA').alias('PAG_QTD_FATURAS_HIST'),
    F.sum('VAL_PAGAMENTO_FATURA').alias('PAG_TOTAL_PAGO_HIST'),
    F.avg('VAL_PAGAMENTO_FATURA').alias('PAG_MEDIA_PAGO_HIST'),

    # --- Métricas de Risco (Atraso) ---
    F.max('DIAS_ATRASO_CALC').alias('PAG_MAX_DIAS_ATRASO_HIST'),
    F.avg('DIAS_ATRASO_CALC').alias('PAG_MEDIA_DIAS_ATRASO_HIST'),
    F.sum('FLAG_ATRASO').alias('PAG_QTD_PAGAMENTOS_ATRASADOS'),
    F.stddev('DIAS_ATRASO_CALC').alias('PAG_DESVIO_PADRAO_ATRASO'),

    # --- Recência ---
    F.max('DAT_STATUS_PAGAMENTO').alias('DATA_ULTIMO_PAGAMENTO')
)

# Cálculo de Recência (Dias)
df_book_pagamento = df_book_pagamento.withColumn(
    'PAG_DIAS_DESDE_ULTIMO_PAG',
    F.datediff(
        F.to_date(F.concat(F.substring(F.col('ID_UNICO'), -6, 6), F.lit('01')), 'yyyyMMdd'),
        F.col('DATA_ULTIMO_PAGAMENTO')
    )
).drop('DATA_ULTIMO_PAGAMENTO')

# Tratamento de Nulos iniciais (Desvio Padrão pode vir nulo se só tiver 1 fatura)
df_book_pagamento = df_book_pagamento.fillna(0, subset=['PAG_DESVIO_PADRAO_ATRASO'])

In [44]:
df_book_pagamento.show(5)

+-----------------+--------------------+-------------------+-------------------+------------------------+--------------------------+----------------------------+------------------------+-------------------------+
|         ID_UNICO|PAG_QTD_FATURAS_HIST|PAG_TOTAL_PAGO_HIST|PAG_MEDIA_PAGO_HIST|PAG_MAX_DIAS_ATRASO_HIST|PAG_MEDIA_DIAS_ATRASO_HIST|PAG_QTD_PAGAMENTOS_ATRASADOS|PAG_DESVIO_PADRAO_ATRASO|PAG_DIAS_DESDE_ULTIMO_PAG|
+-----------------+--------------------+-------------------+-------------------+------------------------+--------------------------+----------------------------+------------------------+-------------------------+
|79NWUX9YNTW202502|                   9|             440.12|  48.90222222222222|                      67|        16.333333333333332|                           6|       23.24865587512534|                       28|
|79XTXZ78XZU202412|                  24|            1343.68|  55.98666666666667|                      22|        0.7083333333333334|                

In [46]:
#-------------------------------
# ARREDONDAR E REMOVER NEGATIVOS
#-------------------------------

cols_pagamento = [
    'PAG_QTD_FATURAS_HIST',
    'PAG_TOTAL_PAGO_HIST',
    'PAG_MEDIA_PAGO_HIST',
    'PAG_MAX_DIAS_ATRASO_HIST',
    'PAG_MEDIA_DIAS_ATRASO_HIST',
    'PAG_QTD_PAGAMENTOS_ATRASADOS',
    'PAG_DESVIO_PADRAO_ATRASO',
    'PAG_DIAS_DESDE_ULTIMO_PAG'
]

for col in cols_pagamento:
    df_book_pagamento = df_book_pagamento.withColumn(
        col,
        F.round(
            F.when(F.col(col) < 0, 0).otherwise(F.col(col)),
            2
        )
    )

In [47]:
get_shape(df_book_pagamento)

('Quanitade de linhas: 667644', 'Quanitade de Colunas: 9')

In [48]:
get_shape(df_gold)

('Quanitade de linhas: 1891934', 'Quanitade de Colunas: 135')

In [51]:
# -----------------------------------------
# REALIZANDO LEFT JOIN E MANTENDO TODA GOLD
#------------------------------------------

df_abt_final = df_gold.join(
    df_book_pagamento,
    on='ID_UNICO',
    how='left'
)

In [52]:
#-------------------------------------------------------------------------------------
# CRIANDO FLAG MISSING
# Lógica: Se 'PAG_QTD_FATURAS_HIST' é Nulo, o cliente não existe no book de pagamentos
#--------------------------------------------------------------------------------------

df_abt_final = df_abt_final.withColumn(
    'PAG_HISTORICO_MISSING',
    F.when(F.col('PAG_QTD_FATURAS_HIST').isNull(), 1).otherwise(0)
)

In [53]:
#-------------------------------------------
# IMPUTAÇÃO DE NULOS COM -1 (NÃO SE APLICA)
# Quem não deu match no join fica com -1
#--------------------------------------------

df_abt_final = df_abt_final.fillna(-1, subset=cols_pagamento)

In [54]:
df_abt_final.show(5)

+-----------------+--------------+-----------+------+--------+-------------+-------+--------+---+--------+----------------+--------+----------------+----------------+---+---+---+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------+---------------------------+------------+---------+-----------------+-------------------------+------------+------------------+--------------------------+-------------+-----------------------+-------------------------------+----------+-----------------+-------------------------+--------------------+------------------------

In [55]:
comparar_datasets(df_gold, df_abt_final, 'FPD')

 ANÁLISE COMPARATIVA: FPD
 DATAFRAME 1 (Referência):
   - Shape: (1891934, 135)
   - Proporção de FPD: 22.40%

 DATAFRAME 2 (Atual):
   - Shape: (1891934, 144)
   - Proporção de FPD: 22.40%

 IMPACTO DA TRANSFORMAÇÃO:
   - Linhas removidas: 0 (0.00% de perda)
   - Variação no Target: 0.0000 p.p. (pontos percentuais)


## Salvamento

In [ ]:
path_silver = "/content/drive/MyDrive/Hackathon_POD/Gold/plus_pagamento.parquet"
df_abt_final.write.mode("overwrite").parquet(path_silver)

## Relatório

1 -  **Metodologia de Construção e Integridade Temporal**
A extração de variáveis de pagamento seguiu uma rigorosa política de Corte Temporal (Anti-Leakage) para evitar o uso de informações futuras durante o treinamento do modelo.

- Regra de Evento Válido: Apenas faturas com `DAT_STATUS_FATURA` estritamente inferior à data da SAFRA do registro foram consideradas.

- Cálculo de Atraso: A variável base de risco, `DIAS_ATRASO_CALC`, foi derivada da diferença entre a data real do pagamento (`DAT_STATUS_PAGAMENTO`) e a data de vencimento acordada (`DAT_VENCIMENTO_CREDITO`).

- Sanitização: Registros com inconsistências sistêmicas (valores negativos resultantes de imputações prévias ou erros de processamento) foram zerados para não enviesar as métricas de tendência e média.

2 - **Dicionário de Variáveis e Relevância para o Risco**

| Variável | Lógica de Construção | Relevância para o Risco de Crédito
| :--- | :--- | :--- |
|`PAG_QTD_FATURAS_HIST`| Contagem total de faturas liquidadas no histórico válido| Indica o tempo de relacionamento e a consistência do cliente com a operadora.
|`PAG_TOTAL_PAGO_HIST`|Soma acumulada dos valores pagos em faturas.|Reflete o volume financeiro transacionado e a relevância do cliente para o faturamento.
|`PAG_MEDIA_PAGO_HIST`|Média do valor das faturas pagas.|Ajuda a estabilizar a visão de renda e compromisso financeiro mensal.
|`PAG_MAX_DIAS_ATRASO_HIST`|O maior valor encontrado na coluna de dias de atraso.|O atraso máximo histórico é um dos principais preditores de inadimplência(Default).
|`PAG_MEDIA_DIAS_ATRASO_HIST`|Média aritmética dos dias de atraso em todas as faturas.|Identifica o comportamento padrão. Diferencia o atraso pontual do atraso crônico.
|`PAG_QTD_PAGAMENTOS_ATRASADOS`|Soma de ocorrências onde o atraso foi maior que zero dias.|"Mede a frequência de impontualidade, indicando hábitos de pagamento de risco."
|`PAG_DESVIO_PADRAO_ATRASO`|Desvio padrão dos dias de atraso.|Avalia a volatilidade do comportamento. Desvios altos indicam instabilidade financeira.
|`PAG_DIAS_DESDE_ULTIMO_PAG`|Diferença entre a Data da Safra e o último pagamento registrado.|Recência: Avalia se o cliente continua ativo no fluxo de pagamentos ou se houve interrupção recente.
|`PAG_HISTORICO_MISSING`|Flag binária (0 ou 1) indicando ausência de dados no book.|"Identifica o segmento de clientes ""Sem Histórico"", essencial para o tratamento de novos entrantes."

3 - **Estratégia de Integração e Tratamento de Dados Ausentes**
Diferente do módulo de recarga, a integração deste book à Gold utilizou a estratégia de Left Join, fundamentada em evidências estatísticas observadas durante a Análise Exploratória de Dados (EDA):

- Viés de Seleção: A análise comparativa revelou que clientes sem histórico de pagamento possuem uma taxa de inadimplência (FPD Médio de 25,38%) significativamente superior aos clientes com histórico (16,92%).

- Manutenção da Massa: O uso do Left Join permitiu reter 100% da população da Gold, evitando o descarte de 1.224.290 registros de alto risco que seriam eliminados em um Inner Join.

- Imputação de Valores: Para o segmento sem histórico, as variáveis métricas foram preenchidas com o valor sentinela -1.